# Strategic Plans Query

This notebook prepares strategic plan PDFs for querying. It loads local files from `docs/strategicplans` and builds a clean in-memory text view for each document.


## Setup

Install required packages in the active environment.


In [ ]:
%pip install -q pypdf tqdm


## Imports


In [ ]:
from pathlib import Path
import hashlib
import re
from typing import Dict, List

from pypdf import PdfReader
from tqdm.auto import tqdm


## Configuration

Paths and file expectations are centralized here for consistent notebook runs.


In [ ]:
PROJECT_ROOT = Path.cwd()
PDF_DIR = PROJECT_ROOT / "docs" / "strategicplans"

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 250
MIN_CHUNK_CHARS = 120

EXPECTED_FILES = {
    "Naples_Federico_Piano_strategico_2021_2026.pdf",
    "Politechnico_di_Milano_2023-2025.pdf",
    "Piano Strategico Luiss 2021-2024 (per sito).pdf",
    "Bocconi_Piano_Strategico2021-2025&Vision2030.pdf",
    "LUM_PIANO-STRATEGICO-DATENEO-2021-2025.pdf",
    "Sapienza_2021_2027.pdf",
}

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF directory: {PDF_DIR}")


## Source Files

List and validate the strategic plan PDF files that will be used for querying.


In [ ]:
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
found_files = {p.name for p in pdf_paths}

print(f"Found {len(pdf_paths)} PDF files")
for path in pdf_paths:
    print(" -", path.name)

missing = EXPECTED_FILES - found_files
extra = found_files - EXPECTED_FILES

if missing:
    raise FileNotFoundError(f"Missing expected strategic plan files: {sorted(missing)}")

if extra:
    print(f"Warning: additional files present and ignored by expectation check: {sorted(extra)}")


## Text Loading

Each strategic plan is read page by page and cleaned into a single text block.


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_document_text(pdf_path: Path) -> Dict:
    reader = PdfReader(str(pdf_path))

    page_texts: List[str] = []
    non_empty_pages = 0

    for page in reader.pages:
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if len(text) < 40:
            continue

        non_empty_pages += 1
        page_texts.append(text)

    full_text = "\n".join(page_texts)

    return {
        "source_file": pdf_path.name,
        "source_path": str(pdf_path),
        "total_pages": len(reader.pages),
        "non_empty_pages": non_empty_pages,
        "text": full_text,
    }


In [ ]:
strategic_docs: List[Dict] = []

for pdf_path in tqdm(pdf_paths, desc="Loading strategic plans"):
    strategic_docs.append(extract_document_text(pdf_path))

print(f"Loaded {len(strategic_docs)} strategic plan documents")

for doc in strategic_docs:
    print(
        f" - {doc['source_file']}: pages={doc['total_pages']}, non_empty_pages={doc['non_empty_pages']}, chars={len(doc['text'])}"
    )

strategic_docs[0] if strategic_docs else "No strategic plans loaded"


## Chunking

Overlapping chunks are created per document to improve retrieval around text boundaries.


In [ ]:
def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

    chunks: List[str] = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + size, text_len)
        chunk = text[start:end].strip()

        if len(chunk) >= MIN_CHUNK_CHARS:
            chunks.append(chunk)

        if end == text_len:
            break

        start = end - overlap

    return chunks


In [ ]:
documents: List[str] = []
metadatas: List[Dict] = []
ids: List[str] = []

for doc in tqdm(strategic_docs, desc="Chunking strategic plans"):
    chunks = chunk_text(doc["text"])

    for chunk_idx, chunk in enumerate(chunks):
        raw_id = f"{doc['source_file']}|{chunk_idx}|{len(chunk)}"
        chunk_id = hashlib.md5(raw_id.encode("utf-8")).hexdigest()

        documents.append(chunk)
        metadatas.append(
            {
                "source_file": doc["source_file"],
                "source_path": doc["source_path"],
                "total_pages": doc["total_pages"],
                "non_empty_pages": doc["non_empty_pages"],
                "chunk_index": chunk_idx,
                "chunk_chars": len(chunk),
            }
        )
        ids.append(chunk_id)

print(f"Created {len(documents)} chunks")
print(f"Unique IDs: {len(set(ids))}")
if documents:
    print("Sample chunk preview:")
    print(documents[0][:500])
